# Feature Engineering Notebook

## Objective

Transform raw HR data into meaningful machine learning features based on insights discovered during EDA.

These features will improve model performance and interpretability while aligning with business understanding of employee attrition.

## Feature Engineering Setup
Load the dataset and prepare it for transformation based on insights from EDA:
- Salary differences
- Tenure-based churn
- Job role risk patterns
- Age lifecycle trends

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv")

df.shape

(1470, 35)

## Feature 1: Salary Bands

Based on EDA, MonthlyIncome shows strong separation between employees who stayed and those who left.

We convert continuous salary into categorical bands for better interpretability and model stability.

In [2]:
df["SalaryBand"] = pd.qcut(df["MonthlyIncome"], q=4, labels=["Low", "Medium", "High", "Very_High"])

df[["MonthlyIncome", "SalaryBand"]].head()

,MonthlyIncome,SalaryBand
0,5993,High
1,5130,High
2,2090,Low
3,2909,Low
4,3468,Medium


## Feature 2: Age Groups

EDA showed higher attrition in younger employees.

We convert age into career-stage segments for better interpretability.

In [3]:
bins = [18, 25, 35, 45, 60]
labels = ["Entry", "Early_Career", "Mid_Career", "Senior"]

df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels)

df[["Age", "AgeGroup"]].head()

,Age,AgeGroup
0,41,Mid_Career
1,49,Senior
2,37,Mid_Career
3,33,Early_Career
4,27,Early_Career


## Feature 3: Tenure Buckets

EDA showed strong early-career attrition patterns.

We convert YearsAtCompany into tenure stages to capture churn timing behavior.

In [4]:
bins = [0, 2, 5, 10, 40]
labels = ["Early_Churn", "Growth", "Stable", "Veteran"]

df["TenureGroup"] = pd.cut(df["YearsAtCompany"], bins=bins, labels=labels)

df[["YearsAtCompany", "TenureGroup"]].head()

,YearsAtCompany,TenureGroup
0,6,Stable
1,10,Stable
2,0,NaN
3,8,Stable
4,2,Early_Churn


## Feature 4: Combined High Risk Flag

We create a composite risk indicator combining:
- Low salary
- Early tenure
- Low satisfaction

This captures multi-factor attrition risk behavior.

In [5]:
df["HighRiskFlag"] = np.where(
    (df["MonthlyIncome"] < df["MonthlyIncome"].median()) &
    (df["YearsAtCompany"] <= 2) &
    (df["JobSatisfaction"] <= 2),
    1,
    0
)

df["HighRiskFlag"].value_counts()

HighRiskFlag
0    1379
1      91
Name: count, dtype: int64

## Feature Cleanup

We remove:
- constant columns
- identifier columns
- redundant features

This ensures model learns meaningful patterns only.

In [6]:
df = df.drop(columns=[
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
])

df.shape

(1470, 35)

In [8]:
df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})